In [92]:
import os
import warnings

import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import KNNImputer

from mappers import to_numeric

warnings.filterwarnings("ignore", category=DeprecationWarning)

In [93]:
root = os.path.abspath(os.path.join(os.path.dirname(__name__), ".."))
path = os.path.join(root, "data", "raw.csv")

dataset = pd.read_csv(filepath_or_buffer=path)

In [83]:
dataset.shape

(7818, 26)

In [94]:
columns = ["Category URL", "Service URL", "Offer URL", "Offer Name", "Owner URL", "Owner Name"]
dataset.drop(columns=columns, inplace=True)

In [85]:
dataset.shape

(7818, 20)

In [95]:
dataset["Owner Completion Rate"] = dataset["Owner Completion Rate"].mask(dataset["Owner Completion Rate"] == "لم يحسب بعد")
dataset["Owner Completion Rate"] = dataset["Owner Completion Rate"].replace("[\%,]", "", regex=True).astype(float)
dataset["Owner Verified"] = dataset["Owner Verified"].astype(int)
dataset["Price"] = dataset["Price"].replace("[\$,]", "", regex=True).astype(float)


dataset = to_numeric(dataset, columns=["Duration", "Offer Response Time", "Owner Response Time"])

dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7818 entries, 0 to 7817
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Category Name          7818 non-null   object 
 1   Service Name           7818 non-null   object 
 2   Offer Stars            7818 non-null   float64
 3   Offer Raters           7818 non-null   int64  
 4   Offer Response Time    6012 non-null   float64
 5   Offer Buyers           7818 non-null   int64  
 6   Pending                7818 non-null   int64  
 7   Price                  7818 non-null   float64
 8   Duration               7818 non-null   int64  
 9   Reviews                7818 non-null   int64  
 10  Available Additions    7818 non-null   int64  
 11  Additions Price        7818 non-null   float64
 12  Owner Verified         7818 non-null   int64  
 13  Owner Level            7818 non-null   object 
 14  Owner Stars            7818 non-null   float64
 15  Owne

In [96]:
top_prices = (dataset.groupby("Owner Level", group_keys=False).apply(lambda x: x.nlargest(1, "Price")))

top_prices["Frequency"] = top_prices.apply(lambda row: dataset[(dataset["Owner Level"] == row["Owner Level"]) & (dataset["Price"] == row["Price"])].shape[0], axis=1)

order = (top_prices[["Owner Level", "Price", "Frequency"]].sort_values(by=["Price", "Frequency"])["Owner Level"].unique())

ordinal = OrdinalEncoder(categories=[order])

dataset["Owner Level"] = ordinal.fit_transform(dataset[["Owner Level"]])

In [97]:
columns = ["Category Name", "Service Name"]

one_hot = OneHotEncoder()

encoded = one_hot.fit_transform(dataset[columns])
encoded = pd.DataFrame(encoded.toarray(), columns=[col.split("_", 1)[-1] for col in one_hot.get_feature_names_out(columns)])

dataset = pd.concat([dataset, encoded], axis=1).drop(columns, axis=1)

In [98]:
dataset.head()

,Offer Stars,Offer Raters,Offer Response Time,Offer Buyers,Pending,Price,Duration,Reviews,Available Additions,Additions Price,...,موضة وجمال,مونتاج فيديو,نسخ احتياطي ونقل استضافة,نصوص إعلانية,نظم المعلومات الجغرافية GIS,واجهات API والتكاملات,وايت بورد,وصف منتجات,ووكومرس,يوكان
0,4.5,141,2.000,176,0,10.0,1,25,5,420.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,4.5,5,0.300,6,0,10.0,2,5,4,40.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,5.0,1,0.667,2,0,10.0,3,1,2,125.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,5.0,25,13.000,33,0,20.0,1,25,3,95.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4.5,11,0.150,12,0,15.0,1,11,3,275.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
